# ISC English — RAG chunking, embeddings, and retrieval

This notebook prepares chapter/topic notes for retrieval: loads notes, chunks text, checks chunk sizes, generates sentence embeddings, saves aligned metadata/vectors, and demonstrates similarity and chapter-filtered retrieval.

**Current prototype settings (as documented in project work):** `sentence-transformers/all-MiniLM-L6-v2`; 384-dimensional embeddings; token-aware chunking was later used with a target of 200 tokens and overlap of 30. Verify that the current notebook cells and saved files use the same configuration before rerunning.

**Known data-quality limitation:** Some note text contains the replacement character `�`. This indicates text corruption in the source and cannot be repaired reliably by simply deleting the character. Inspect and correct the original notes before generating final embeddings.

In [ ]:
pip install -U langchain-text-splitters

In [ ]:
pip install mysql-connector-python pandas python-dotenv

In [ ]:
import mysql.connector
import pandas as pd

from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import json
import pandas as pd

with open("isc_chapter_notes.json", "r", encoding="utf-8") as file:
    data = json.load(file)

df = pd.DataFrame(data)

print("Total records:", len(df))
print(df.head())

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

print("Text splitter initialized successfully!")

In [ ]:
all_chunks = []

for _, row in df.iterrows():

    note_text = row["chapter_notes"]

    chunks = text_splitter.split_text(note_text)

    for i, chunk in enumerate(chunks):

        all_chunks.append({
            "chapter_note_id": row["chapter_note_id"],
            "chapter_or_topic": row["chapter_or_topic"],
            "note_path": row["note_path"],
            "chunk_id": i,
            "chunk_text": chunk
        })

print("Total chunks generated:", len(all_chunks))

In [ ]:
chunks_df = pd.DataFrame(all_chunks)

print("Total chunks:", len(chunks_df))
print("\nFirst 5 chunks:")

chunks_df.head()

In [ ]:
chunks_df["chunk_length"] = chunks_df["chunk_text"].apply(len)

print(chunks_df["chunk_length"].describe())

In [ ]:
short_chunks = chunks_df[
    chunks_df["chunk_length"] < 100
]

print("Number of short chunks:", len(short_chunks))

short_chunks[
    ["chapter_or_topic", "chunk_id", "chunk_length", "chunk_text"]
].head(20)

In [ ]:
chunk_distribution = (
    chunks_df.groupby("chapter_or_topic")
    .size()
    .reset_index(name="number_of_chunks")
    .sort_values("number_of_chunks", ascending=False)
)

print(chunk_distribution)

In [ ]:
# Remove chunks containing only Markdown separators
chunks_df = chunks_df[
    chunks_df["chunk_text"].str.strip() != "---"
].copy()

# Reset index
chunks_df.reset_index(drop=True, inplace=True)

print("Chunks after removing separators:", len(chunks_df))

In [ ]:
chunks_df[
    chunks_df["chunk_text"].str.contains(
        "True role models",
        case=False,
        na=False
    )
][["chapter_or_topic", "chunk_text"]]

In [ ]:
composition_chunks = chunks_df[
    (chunks_df["chapter_or_topic"] == "Composition") &
    (chunks_df["chunk_id"].between(19, 23))
]

for _, row in composition_chunks.iterrows():
    print(f"\nChunk ID: {row['chunk_id']}")
    print(row["chunk_text"])
    print("-" * 80)

In [ ]:
# Remove chunks containing only Markdown separators
chunks_df = chunks_df[
    chunks_df["chunk_text"].str.strip() != "---"
].copy()

# Remove unnecessary leading full stops from chunks
chunks_df["chunk_text"] = (
    chunks_df["chunk_text"].str.strip().str.lstrip(".").str.strip()
)

# Recalculate chunk lengths
chunks_df["chunk_length"] = chunks_df["chunk_text"].apply(len)

# Reset index
chunks_df.reset_index(drop=True, inplace=True)

print("Total cleaned chunks:", len(chunks_df))

In [ ]:
chunks_df.to_csv(
    "isc_cleaned_chunks.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Cleaned chunks saved successfully!")

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Model loaded successfully!")
print("Embedding dimensions:", model.get_sentence_embedding_dimension())
print("Maximum sequence length:", model.max_seq_length)

In [ ]:
tokenizer = model.tokenizer

chunks_df["token_count"] = chunks_df["chunk_text"].apply(
    lambda text: len(tokenizer.encode(text, add_special_tokens=True))
)

print("Total chunks:", len(chunks_df))
print("Maximum tokens:", chunks_df["token_count"].max())
print("Chunks exceeding 256 tokens:",
      (chunks_df["token_count"] > 256).sum())

chunks_df["token_count"].describe()

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

token_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer,
    chunk_size=200,
    chunk_overlap=30,
    separators=["\n\n", "\n", ". ", " ", ""]
)

all_chunks = []

for _, row in df.iterrows():

    note_text = row["chapter_notes"]

    chunks = token_splitter.split_text(note_text)

    for i, chunk in enumerate(chunks):

        all_chunks.append({
            "chapter_note_id": row["chapter_note_id"],
            "chapter_or_topic": row["chapter_or_topic"],
            "note_path": row["note_path"],
            "chunk_id": i,
            "chunk_text": chunk
        })

chunks_df = pd.DataFrame(all_chunks)

print("Total chunks:", len(chunks_df))

In [ ]:
chunks_df["token_count"] = chunks_df["chunk_text"].apply(
    lambda text: len(tokenizer.encode(
        text,
        add_special_tokens=True
    ))
)

print("Total chunks:", len(chunks_df))
print("Maximum tokens:", chunks_df["token_count"].max())

print(
    "Chunks exceeding 256 tokens:",
    (chunks_df["token_count"] > 256).sum()
)

chunks_df["token_count"].describe()

In [ ]:
import numpy as np

# Extract text from all chunks
texts = chunks_df["chunk_text"].tolist()

# Generate embeddings
embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding generation completed!")
print("Embedding shape:", embeddings.shape)

In [ ]:
import numpy as np

np.save("isc_embeddings.npy", embeddings)

chunks_df.to_csv(
    "isc_embedded_chunks.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Files saved successfully!")

In [ ]:
from google.colab import files

files.download("isc_embeddings.npy")
files.download("isc_embedded_chunks.csv")

In [ ]:
short_chunks = chunks_df[
    chunks_df["token_count"] < 30
][["chapter_or_topic", "chunk_id", "token_count", "chunk_text"]]

print("Number of short chunks:", len(short_chunks))

short_chunks

In [ ]:
import numpy as np
import pandas as pd

embeddings = np.load("isc_embeddings.npy")

chunks_df = pd.read_csv("isc_embedded_chunks.csv")

print("Embeddings shape:", embeddings.shape)
print("Metadata shape:", chunks_df.shape)

In [ ]:
def retrieve_similar_chunks(query, top_k=5):

    # Convert query into an embedding
    query_embedding = model.encode(
        query,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # Calculate cosine similarity
    similarities = np.dot(
        embeddings,
        query_embedding
    )

    # Get indices of top matching chunks
    top_indices = np.argsort(similarities)[::-1][:top_k]

    # Prepare results
    results = chunks_df.iloc[top_indices].copy()

    results["similarity_score"] = similarities[top_indices]

    return results[
        [
            "chapter_or_topic",
            "chunk_id",
            "similarity_score",
            "chunk_text"
        ]
    ]

In [ ]:
query = "Explain Macbeth's ambition and how it leads to his downfall."

results = retrieve_similar_chunks(query, top_k=5)

results

In [ ]:
query = "Explain the central theme and message of The Medicine Bag."

results = retrieve_similar_chunks(query, top_k=5)

results

In [ ]:
def retrieve_by_chapter(query, chapter, top_k=5):

    # Filter chunks belonging to the specified chapter
    filtered_indices = chunks_df[
        chunks_df["chapter_or_topic"].str.lower() == chapter.lower()
    ].index

    if len(filtered_indices) == 0:
        print("Chapter not found!")
        return None

    # Generate query embedding
    query_embedding = model.encode(
        query,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # Calculate similarity only for filtered chunks
    similarities = np.dot(
        embeddings[filtered_indices],
        query_embedding
    )

    # Get top matching chunks
    top_positions = np.argsort(similarities)[::-1][:top_k]

    # Map positions back to original dataframe indices
    top_indices = filtered_indices.to_numpy()[top_positions]

    results = chunks_df.loc[top_indices].copy()

    results["similarity_score"] = similarities[top_positions]

    return results[
        [
            "chapter_or_topic",
            "chunk_id",
            "similarity_score",
            "chunk_text"
        ]
    ]

In [ ]:
query = "Explain Macbeth's ambition and how it leads to his downfall."

results = retrieve_by_chapter(
    query=query,
    chapter="Macbeth",
    top_k=5
)

results

In [ ]:
def build_context(query, chapter, top_k=5):

    results = retrieve_by_chapter(
        query=query,
        chapter=chapter,
        top_k=top_k
    )

    if results is None or results.empty:
        return ""

    context = ""

    for i, row in enumerate(results.itertuples(), start=1):

        context += f"""
        Context {i}
        Chapter: {row.chapter_or_topic}
        Similarity: {row.similarity_score:.4f}

        Content:
        {row.chunk_text}

        -------------------------
        """

    return context

In [ ]:
query = "Explain Macbeth's ambition and how it leads to his downfall."

context = build_context(
    query=query,
    chapter="Macbeth",
    top_k=5
)

print(context)

In [ ]:
corrupted_chunks = chunks_df[
    chunks_df["chunk_text"].str.contains("�", na=False)
]

print("Chunks containing corrupted characters:",
      len(corrupted_chunks))

corrupted_chunks[
    ["chapter_or_topic", "chunk_id", "chunk_text"]
].head(10)

In [ ]:
# Count corrupted chunks in each chapter/topic

corruption_report = (
    chunks_df[
        chunks_df["chunk_text"].str.contains("�", na=False)
    ]
    .groupby("chapter_or_topic")
    .size()
    .reset_index(name="corrupted_chunks")
    .sort_values("corrupted_chunks", ascending=False)
)

print(corruption_report.to_string(index=False))